# Apache Spark Koalas

Este notebook contiene las funciones principales de Koalas, obtenidas de la documentación oficial de https://koalas.readthedocs.io/

In [ ]:
#!pip install plotly

In [6]:
import sys
sys.executable

'/opt/homebrew/anaconda3/envs/ProcesBigData/bin/python'

### Importar librerias

In [36]:
import findspark
findspark.init()

import pyspark
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession

In [35]:
import databricks.koalas as ks

### 1. Creación de objetos



Creando una serie Koalas pasando una lista de valores, permitiendo que Koalas cree un índice entero predeterminado:

In [ ]:
s = ks.Series([1, 3, 5, np.nan, 6, 8])

In [ ]:
s

Creando un Koalas DataFrame pasando un dict de objetos que se pueden convertir a series.

In [ ]:
kdf = ks.DataFrame(
    {'a': [1, 2, 3, 4, 5, 6],
     'b': [100, 200, 300, 400, 500, 600],
     'c': ["one", "two", "three", "four", "five", "six"]},
    index=[10, 20, 30, 40, 50, 60])

### Muestra el dataframe de koalas

In [ ]:
kdf

Creando un DataFrame de pandas pasando una matriz numpy, con un índice de fecha y hora y columnas etiquetadas:

### Usando pandas se crea un dataframe que va a crear un rango de fechas a partir de la primera fecha que se le paso y seran 6 fechas ya que se determino que serian 6 periodos.

In [8]:
dates = pd.date_range('20130101', periods=6)

### Para ver que fechas se generaron en el dataframe

In [9]:
dates

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06'],
              dtype='datetime64[ns]', freq='D')

### Se crea un DataFrame de pandas llamado pdf con 6 filas, 4 columnas, y números aleatorios. El índice es una serie de fechas (antes creadas), y las columnas están etiquetadas como 'A', 'B', 'C', y 'D'

In [10]:
pdf = pd.DataFrame(np.random.randn(6, 4), index=dates, columns=list('ABCD'))

### Se esta mostrando el dataframe

In [11]:
pdf

,A,B,C,D
2013-01-01,-0.635597,-1.661895,-1.278011,1.741264
2013-01-02,-1.619532,-0.197810,-0.812331,-1.196373
2013-01-03,-0.306206,-0.737907,0.414353,-0.236456
2013-01-04,1.106244,1.262021,-0.853152,-0.524182
2013-01-05,2.188626,0.799155,-0.949356,-0.242143
2013-01-06,-0.051320,-0.076941,-0.810506,-1.064698


Ahora, este DataFrame de pandas se puede convertir en un DataFrame de Koalas

### Se esta convirtiendo un Dataframe de pandas a koalas con la funcion ks.from_pandas()


In [ ]:
kdf = ks.from_pandas(pdf)

### Para conocer el tipo de variable

In [ ]:
type(kdf)

Sin embargo, se ve y se comporta igual que un DataFrame de pandas

### Mostrando como se ve el Dataframe de koalas para compararlo visualmente contra el dataframe de pandas(unas celdas antes)

In [ ]:
kdf

Además, es posible crear un **Koalas DataFrame desde Spark DataFrame**.

Creando un Spark DataFrame a partir de pandas DataFrame

### Se está creando o obteniendo una instancia de SparkSession, que es el punto de entrada principal para trabajar con Spark en PySpark

In [12]:
spark = SparkSession.builder.getOrCreate()

24/10/28 11:36:21 WARN Utils: Your hostname, Gretels-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.18 instead (on interface en0)
24/10/28 11:36:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/10/28 11:36:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Convierte el DataFrame de pandas llamado pdf a un DataFrame de Spark, asignándolo a la variable sdf

In [13]:
sdf = spark.createDataFrame(pdf)

### Para mostrar un dateframe de spark

In [14]:
sdf.show()

+--------------------+--------------------+-------------------+--------------------+
|                   A|                   B|                  C|                   D|
+--------------------+--------------------+-------------------+--------------------+
| -0.6355972747115829| -1.6618946349141206|-1.2780109722944204|  1.7412640492361908|
| -1.6195323752542015| -0.1978102082464226|-0.8123312725424994|  -1.196373184294278|
|  -0.306205980933192| -0.7379074296517587| 0.4143534087471814|-0.23645635905544804|
|  1.1062443847090437|   1.262021234266452|-0.8531523791762299|  -0.524182403508425|
|   2.188626242077268|  0.7991547268895383|-0.9493555018185709| -0.2421425321678524|
|-0.05131952542664316|-0.07694076754327488|-0.8105064166961152| -1.0646981732157326|
+--------------------+--------------------+-------------------+--------------------+



Creando Koalas DataFrame desde Spark DataFrame.
`to_koalas ()` se adjunta automáticamente a Spark DataFrame y está disponible como una API cuando se importa Koalas.

### convierte un DataFrame de Spark (sdf) a un DataFrame de Koalas 

In [16]:
kdf = sdf.to_koalas()

In [ ]:
kdf

Tiene [dtypes] específicos. Actualmente se admiten los tipos que son comunes a Spark y pandas.

### Permitirá ver los tipos de datos de cada columna en el DataFrame kdf

In [ ]:
kdf.dtypes

### 2. Manipulación de datos


A diferencia de los pandas, los datos en un dataframe de datos de Spark no están _ordenados_, no tienen una noción intrínseca de índice. Cuando se le solicite el encabezado, Spark solo tomará el número solicitado de filas de una partición. **No hay que utilizar el df de Koalas para devolver filas específicas**, use `.loc` o` iloc` en su lugar.

### permite ver 5 filas del DataFrame (no necesariamente regresa las primeras lineas)

In [ ]:
kdf.head()

Muestre el índice, las columnas y los datos numéricos subyacentes.

También puede recuperar el índice; la columna de índice se puede atribuir a un DataFrame, ver más adelante

In [ ]:
kdf.index

### Para que se regrese el nombre de las columnas

In [ ]:
kdf.columns

### Se esta convirtiendo el Dataframe a un arreglo.


In [ ]:
kdf.to_numpy()

**Describe** muestra un resumen estadístico rápido de sus datos

In [ ]:
kdf.describe()

Transposición de sus datos

### Para trasponer la informacion (cambiar filas por columnas y viceversa)

In [ ]:
kdf.T

Ordenando por su índice

### .sort_index por default ordena del primero al ultimo, al agregar el False lo ordena al reves


In [ ]:
kdf.sort_index(ascending=False)

Ordenar por valor

### .sort_values ordena los valores, en este caso se esta ordenando tomando en cuenta los valores de la columna B

In [ ]:
kdf.sort_values(by='B')

### 3. Datos faltantes
Koalas utiliza principalmente el valor `np.nan` para representar los datos faltantes. Por defecto, no se incluye en los cálculos.


### Se está tomando un DataFrame de pandas y lo está reorganizando o redimensionando, se crea un nuevo DataFrame pdf1 con las primeras cuatro fechas como filas y con las columnas originales de pdf más una columna adicional llamada 'E', que tendrá valores NaN en todos los registros iniciales

In [21]:
pdf1 = pdf.reindex(index=dates[0:4], columns=list(pdf.columns) + ['E'])
pdf1

,A,B,C,D,E
2013-01-01,-0.635597,-1.661895,-1.278011,1.741264,NaN
2013-01-02,-1.619532,-0.197810,-0.812331,-1.196373,NaN
2013-01-03,-0.306206,-0.737907,0.414353,-0.236456,NaN
2013-01-04,1.106244,1.262021,-0.853152,-0.524182,NaN


### Se está asignando el valor 1 a la columna 'E' del dataframe, pero solo para las filas correspondientes a las fechas entre dates[0] y dates[1]

In [22]:
pdf1.loc[dates[0]:dates[1], 'E'] = 1
pdf1

,A,B,C,D,E
2013-01-01,-0.635597,-1.661895,-1.278011,1.741264,1.0
2013-01-02,-1.619532,-0.197810,-0.812331,-1.196373,1.0
2013-01-03,-0.306206,-0.737907,0.414353,-0.236456,NaN
2013-01-04,1.106244,1.262021,-0.853152,-0.524182,NaN


### está convirtiendo el DataFrame de pandas pdf1 en un DataFrame de Koalas y asignándolo a kdf1


In [20]:
kdf1 = ks.from_pandas(pdf1)

In [24]:
kdf1

Para eliminar las filas que tienen datos faltantes.

### eliminando de kdf1 todas las filas que contengan al menos un valor NaN

In [ ]:
kdf1.dropna(how='any')

Llenando los datos faltantes.

### De lo que no tenga valor se va a llenar con valor de 5 en esas celdas

In [ ]:
kdf1.fillna(value=5)

### 4. Operaciones

#### Estadísticas
Las operaciones en general excluyen los datos faltantes.

Realización de una estadística descriptiva:

In [28]:
kdf.mean()

#### Configuraciones de Spark

Varias configuraciones en PySpark se pueden aplicar internamente en Koalas.
Por ejemplo, puede habilitar la optimización de Arrow para acelerar enormemente la conversión de pandas internos.

### prev obtiene el valor actual de la configuración de Spark que habilita las optimizaciones de Arrow para la ejecución de consultas. Se guarda en prev para referenciarla más adelante
### ks.set_option establece el tipo de índice por defecto en Koalas como "distribuido".
### y luego se ignoran los warnings 

In [37]:
prev = spark.conf.get("spark.sql.execution.arrow.enabled")  # Keep its default value.
ks.set_option("compute.default_index_type", "distributed")  # Use default index prevent overhead.

import warnings
warnings.filterwarnings("ignore")  # Ignore warnings coming from Arrow optimizations.

In [46]:
spark.conf.set("spark.sql.execution.arrow.enabled", True)
%timeit ks.range(300000).to_pandas()

In [ ]:
spark.conf.set("spark.sql.execution.arrow.enabled", False)
%timeit ks.range(300000).to_pandas()

In [48]:
ks.reset_option("compute.default_index_type")
spark.conf.set("spark.sql.execution.arrow.enabled", prev)  # Set its default value back.

#### Agrupación
Por "agrupar por" nos referimos a un proceso que involucra uno o más de los siguientes pasos:

- Dividir los datos en grupos según algunos criterios.
- Aplicar una función a cada grupo de forma independiente
- Combinar los resultados en una estructura de datos

In [49]:
kdf = ks.DataFrame({'A': ['foo', 'bar', 'foo', 'bar',
                          'foo', 'bar', 'foo', 'foo'],
                    'B': ['one', 'one', 'two', 'three',
                          'two', 'two', 'one', 'three'],
                    'C': np.random.randn(8),
                    'D': np.random.randn(8)})

In [ ]:
kdf

Agrupar y luego aplicar el **sum** a los grupos resultantes.

In [ ]:
kdf.groupby('A').sum()

In [ ]:
kdf.groupby(['A', 'B']).sum()

### 5. Visualización de datos: Generar gráficos


### Creando una serie de pandas llamada pser que contiene 1000 valores aleatorios generados a partir de una distribución normal, tambien se crea un índice de fechas que comienza el 1 de enero de 2000 y tiene una duración de 1000 períodos (días consecutivos). Esto genera un rango de fechas que se utilizará como índice para la serie

In [44]:
pser = pd.Series(np.random.randn(1000),
                 index=pd.date_range('1/1/2000', periods=1000))
pser

2000-01-01    0.993883
2000-01-02   -0.516379
2000-01-03    0.423578
2000-01-04   -1.032731
2000-01-05   -1.230394
                ...   
2002-09-22   -0.941604
2002-09-23   -1.009972
2002-09-24    0.201479
2002-09-25    0.738925
2002-09-26    0.450602
Freq: D, Length: 1000, dtype: float64

### Creando una nueva serie de Koalas a partir de la serie de pandas llamada pser

In [43]:
kser = ks.Series(pser)

### Devuelve el valor máximo de todos los elementos anteriores hasta ese punto

In [ ]:
kser = kser.cummax()
kser

**Gráficos de matplotlib**

### Generar un grafico de lineas con la libreria de matplotlib 

In [ ]:
ks.options.plotting.backend = "matplotlib"

kser.plot()

**Gráficos de Pandas Bokeh**

### Se genera un grafico con bokeh, hacer un gráfico interactivo que muestra los valores acumulativos a lo largo del tiempo, donde puedes hacer zoom, desplazarte y obtener más información al pasar el mouse sobre los puntos del gráfico.

In [ ]:
ks.options.plotting.backend = "pandas_bokeh"

kdf.plot(backend="pandas_bokeh", title="Example Figure")
kser.plot()


**Gráfico por defecto con plotly**

In [ ]:
kser.plot()

En un DataFrame, el plot() es una conveniencia para trazar todas las columnas con etiquetas:

In [45]:
pdf = pd.DataFrame(np.random.randn(1000, 4), index=pser.index,
                   columns=['A', 'B', 'C', 'D'])

In [71]:
kdf = ks.from_pandas(pdf)

### Devuelve el valor máximo de todos los elementos anteriores hasta ese punto

In [ ]:
kdf = kdf.cummax()
kdf

In [ ]:
kdf.plot()

### 6. Entrada / salida de datos


### CSV

CSV es sencillo y fácil de usar

### Convertir el dataframer en csv y luego leer las primeras 10 lineas

In [ ]:
kdf.to_csv('foo.csv')
ks.read_csv('foo.csv').head(10)

### Parquet

Parquet es un formato de archivo eficiente y compacto para leer y escribir más rápido.

### se convierte el archivo en parquet que es un formato de almacenamiento de datos,  permite recuperar los datos de manera eficiente

In [ ]:
kdf.to_parquet('bar.parquet')
ks.read_parquet('bar.parquet').head(10)

### Spark IO

Además, Koalas es totalmente compatible con las diversas fuentes de datos de Spark, como ORC y una fuente de datos externa.

### guardar un DataFrame de Koalas (kdf) en un formato ORC (Optimized Row Columnar) y luego leer ese archivo ORC en un DataFrame de Koalas

In [ ]:
kdf.to_spark_io('zoo.orc', format="orc")
ks.read_spark_io('zoo.orc', format="orc").head(10)